# ML Project / NASDAQ Prediction and Analysis

**Objective** : Predicting possible long/short entries on the NASDAQ index, using Machine Learning algorithms and other tools to carry this project.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import requests 
import io
import os

In [ ]:
path_to_data = ("./data")

## Data

We take the current Nasdaq-100 list from Wikipedia. This means we use today's companies for the whole period, so there's some survivorship bias (we'll mention it in the report).

In [ ]:
html = requests.get("https://en.wikipedia.org/wiki/List_of_NASDAQ-100_companies", headers={"User-Agent": "Mozilla/5.0"}).text
table = pd.read_html(io.StringIO(html))
ndx = next(t for t in table if "Ticker" in t.columns)
TICKERS = ndx["Ticker"].str.replace(".", "-", regex=False).tolist()

In [ ]:
TICKERS

On top of the stocks we add some macro data: volatility indices (VIX, VXN), treasury yields, oil, gold, silver, the dollar and Bitcoin. The idea is to give the model some context about the market, not just prices.

In [ ]:
NASDAQ_FEATURES = {
    # --- INDICES ET VOLATILITÉ ---                    
    "^IXIC": "Nasdaq_composite",              # Broad tech market index
    "^VXN": "Nasdaq_volatility",              # Nasdaq-100 implied volatility (Tech fear index)
    "^VIX": "Volatility",                     # S&P 500 implied volatility (Market fear index)

    # --- interest ---
    "^IRX": "3month_treasury_yield",          # 3-month Treasury yield
    "^FVX": "5y_treasury_yield",              # 5-year Treasury yield
    "^TNX": "10y_treasury_yield",             # 10-year Treasury yield (Tech discounting rate)
    "^TYX": "30y_treasury_yield",             # 30-year Treasury yield

    # --- MATIÈRES PREMIÈRES ET DEVISES ---
    "DX-Y.NYB": "US_dollar",                  # US dollar index (DXY)
    "CL=F": "crude_oil",                      # WTI crude oil futures
    "GC=F": "gold",                           # Gold futures
    "SI=F": "silver",                         # Silver futures
    "BTC-USD": "Bitcoin",                     # Bitcoin USD (Risk-on sentiment proxy)
}
tickers1 = list(NASDAQ_FEATURES.keys())

In [ ]:
MACRO = {
    "^VIX": "Volatility",       # implied volatility (fear index)
    "^TNX": "10y_treasury_yield",       # 10-year Treasury yield
    "^IRX": "3month_treasury_yield",       # 3-month Treasury yield
    "CL=F": "crude_oil",       # WTI crude oil futures
    "DX-Y.NYB": "US_dollar",   # US dollar index
    "GC=F": "gold", 
    "SI=F": "silver",  # gold futures
}
tickers2 = list(MACRO.keys())

In [ ]:
TICKERS = list(dict.fromkeys(TICKERS + tickers1 + tickers2))

In [ ]:
df_nasdaq = yf.download("^NDX", start="2018-01-01", end="2026-09-30",
                     auto_adjust=True, progress=False)[['Open', 'Close','Volume']]
print(df_nasdaq.columns)
print(df_nasdaq.tail())

In [ ]:
df_nasdaq.head(20)

In [ ]:
raw = yf.download(TICKERS, start="2018-01-01", end="2026-09-30")[['Open', 'Close','Volume']]

In [ ]:
raw.head(10)

We join everything on the Nasdaq-100 dates (inner join). Bitcoin trades on weekends but the index doesn't, so this way we only keep real trading days and we don't need to fill anything.

In [ ]:
raw = pd.concat((df_nasdaq,raw),axis=1,join='inner',)

In [ ]:
raw.loc[:, (slice(None), 'BTC-USD')]

### Missing values

Some companies are very recent (ARM, CRWV, SNDK...), so they have almost no history. We drop the columns with less than 40% of data. We picked this threshold by looking at which columns were left.

We also drop GOOG (almost the same as GOOGL) and ^IXIC (it moves with the Nasdaq-100), they don't add anything new.

In [ ]:
dropping = raw.isnull().sum().sort_values(ascending=False)/len(raw)

In [ ]:
dropping

In [ ]:
raw_drop = raw.dropna(axis=1, thresh=int(0.4 * len(raw)))
# GOOG is basically GOOGL and ^IXIC moves with ^NDX, so they add nothing
raw_drop = raw_drop.drop(columns=["GOOG", "^IXIC"], level=1)

In [ ]:
print(list(set(raw.columns.tolist()) - set(raw_drop.columns.tolist())))

In [ ]:
try:
    raw_drop.loc[:, (slice(None), "SPCX")]
except KeyError:
    print("SPCX not in raw_drop: it was dropped")

In [ ]:
raw.loc[:, (slice(None), "SPCX")]

In [ ]:
raw_drop.describe()

In [ ]:
from fastai.tabular.all import add_datepart

raw_drop = raw_drop.reset_index()
raw_time = add_datepart(raw_drop, 'Date', drop=False)

In [ ]:
raw_time.isnull().sum().sort_values(ascending=False)/len(raw)

In [ ]:
columns = raw_time.columns.to_list()
print(columns)

In [ ]:
raw_time = raw_time.copy(True)

In [ ]:
raw_time.loc[:, (slice(None),'^NDX')]

## Target

We decide at the close of day `t`, buy at the open of `t+1` and sell:
- at the close of `t+1` (1-day horizon)
- at the close of `t+5` (5-day horizon)

We use log returns, and the label is 1 if the return is positive, 0 if not. The negative shifts are fine here because they are only used for the target, never for the features.

In [ ]:
ndx = raw_drop.set_index('Date')
close = ndx['Close']['^NDX']
open_ = ndx['Open']['^NDX']

entry  = open_.shift(-1)     
exit_1 = close.shift(-1)     
exit_5 = close.shift(-5)     

In [ ]:
open_

In [ ]:
entry

In [ ]:
ndx_1 = np.log(exit_1 / entry)
ndx_2 = np.log(exit_5 / entry)
#(ndx["fwd_ret"] > 0).astype(float).where(ndx["fwd_ret"].notna())

In [ ]:
ndx_1

In [ ]:
raw_time = raw_time.set_index("Date")

In [ ]:
raw_time

In [ ]:
data = raw_time.copy()
# both have the Date index, so pandas aligns them by date
data["fwd_ret_1d"] = ndx_1
data["fwd_ret_5d"] = ndx_2

In [ ]:
for h in ("1d", "5d"):
    r = data[f"fwd_ret_{h}"]
    data[f"y_{h}"] = (r > 0).astype(float).where(r.notna())

In [ ]:
print(data[["fwd_ret_1d", "fwd_ret_5d", "y_1d", "y_5d"]].tail(7))

In [ ]:
print(data.columns.tolist())
print(data.index.min(), data.index.max(), len(data))

The last rows don't have future prices, so they have no target. We remove them.

In [ ]:
target_cols = [("fwd_ret_1d", ""), ("fwd_ret_5d", ""), ("y_1d", ""), ("y_5d", "")]

data = data.dropna(subset=[("y_1d", ""), ("y_5d", "")])

In [ ]:
data.to_feather("data/final_data.feather")

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

## Train / test split

The test set starts in July 2025 and we don't touch it until the end.

The 5-day target uses prices up to `t+5`, so the last days of train would "see" the test period. That's why we remove the last 6 rows of train, and we use the same gap in `TimeSeriesSplit`.

In [ ]:
TEST_START = "2025-07-01"
train = data.loc[:"2025-06-30"]
test = data.loc[TEST_START:]

In [ ]:
gap = 6
train = train.iloc[:-gap]

In [ ]:
print("train:", train.index.min(), "->", train.index.max(), len(train))
print("test: ", test.index.min(), "->", test.index.max(), len(test))

In [ ]:
test.to_feather("data/test_holdout.feather")

In [ ]:
tscv = TimeSeriesSplit(n_splits=5, gap=gap)
X_train = train.drop(columns=target_cols)
y_train = train[[("y_5d", ""),("y_1d", "")]]

Class balance: in train the market goes up 54% of the days (1d) and 58% of the time over 5 days. So a model that always says "up" already gets that accuracy, that's the baseline to beat.

In [ ]:
y_train.describe()

## EDA (only on train)

The 1-day returns have fat tails (kurtosis around 5.9) and are almost symmetric. The 5-day returns have negative skew (-0.58): the big drops are bigger than the big rises.

In [ ]:
train[[("fwd_ret_1d", ""), ("fwd_ret_5d", "")]].hist(  bins=60, figsize=(12, 5), layout=(1, 2), edgecolor="black")
plt.tight_layout()
plt.show()

In [ ]:
train[[("fwd_ret_1d", ""), ("fwd_ret_5d", "")]].kurtosis()

In [ ]:
train[[("fwd_ret_1d", ""), ("fwd_ret_5d", "")]].skew()

### Volatility

Realized volatility (21 days, annualized) with past close-to-close returns, compared with the VXN (implied volatility). The VXN is usually a bit higher (24% vs 22% on average) and both jump in the stress periods like 2020 and 2022.

In [ ]:
# past close-to-close returns, so the value at t only uses data up to the close of t
ndx_ret = np.log(train[("Close", "^NDX")]).diff()
vol_21d = ndx_ret.rolling(21).std()*np.sqrt(252)*100
vxn = train[("Close", "^VXN")]

fig, ax = plt.subplots(figsize=(20,8))

ax.plot(vxn.index, vxn.values, label="VXN (implied, 30d)", color="tab:red", linewidth=1)
ax.plot(vol_21d.index, vol_21d.values, label="Realized vol (21d, annualized)",color="tab:blue", linewidth=1)

ax.set_title("Implied vs realized volatility, Nasdaq-100 context")
ax.set_ylabel("Annualized volatility (%)")
ax.legend()
ax.grid(alpha=0.3)

plt.show()

In [ ]:
volatility = {"Realized vol (21d)": vol_21d, "VXN": vxn}
nasdaq = df_nasdaq['Open','^NDX']

for name, vol in volatility.items():
    nasdaq1 = nasdaq.loc[vol.index]
    fig, ax1 = plt.subplots(figsize=(20,8))

    ax2 = ax1.twinx()
    ax2.plot(nasdaq1.index, nasdaq1.values, color="green", alpha=0.35, linewidth=1, label="Nasdaq-100")
    ax2.set_ylabel("Nasdaq-100 level")
    ax2.set_zorder(ax1.get_zorder() - 1)   
    ax1.patch.set_visible(False) 

    ax1.plot(vol.index, vol.values, color="tab:red", linewidth=1, label=name)
    ax1.set_ylabel(name)
    ax1.set_title(f"{name} over the Nasdaq-100")

    lines = ax1.get_lines() + ax2.get_lines()
    ax1.legend(lines, [l.get_label() for l in lines], loc="upper left")

    ax1.grid(alpha=0.3)
    plt.tight_layout()

plt.show()



In [ ]:
nasdaq.loc[train.index].plot(figsize=(20, 8), color="tab:red", label="Nasdaq-100 (open)")
plt.legend()